# The late file, and why readiness is a queue

§5.1.1 says the upload file "usually arrives before the bag", and §5.2.2 spends
that: geocoding has already run, so it costs the hub nothing. The exception is
one sentence in §5.2.5 — "where the file is late or missing (§5.1.1), geocoding
cannot have run ahead of the bag and joins the sequence after reconciliation" —
and this notebook is that sentence, in both of its halves.

The second half of the notebook is about the word **queue**. §5.2.5 reads like
a sum of delays, and Open Question 4 asks for each step in "envelopes per hour",
which is a server working through a line rather than a delay each envelope
suffers privately. The difference is the whole of §8.3.

**Every number here is provisional.** All four throughputs are invented and
`docs/assumptions.md` labels them so. What this notebook shows is the *shape*
the rates produce; the rates themselves are placeholders, and
`docs/capacity-finding.md` is the standing warning about the difference.

In [ ]:
from ddn import assumptions, processing

HOUR = 3600
VAN_BACK = 17 * HOUR  # §5.2.5 starts the sum at the van's expected return.

for step, rate in (("§5.2.1 reconcile", assumptions.RECONCILE_PER_HOUR),
                   ("§5.2.2 geocode", assumptions.GEOCODE_PER_HOUR),
                   ("§5.2.3 clean room", assumptions.ASSEMBLY_PER_HOUR),
                   ("§5.2.4 sort", assumptions.SORT_PER_HOUR)):
    print(f"{step:<20}{rate:>6} envelopes/hour  ({3600 / rate:>5.1f} s each)")

## Three ways a file can fail to beat its bag — and one that is not one

§9.1 records `file_received_at` on the mailbag, and `late_file_bags` reads
§5.1.1's "late **or missing**" off it. The two halves are different facts and
the code keeps them apart: a null says the file never came, a time after the
bag's arrival says it came too late to have helped.

The fourth bag below carries no such field at all, and that is **not** late.
§9.1 makes the field optional, and an absent optional field says nothing was
recorded — a different claim from a recorded null. Reading silence as "no file
came" would put every bag predating the field on the slow path, which is a
different day from the one that happened.

In [ ]:
requests = [
    {"mailbag_id": "B1", "file_received_at": 9 * HOUR},   # the usual case
    {"mailbag_id": "B2", "file_received_at": None},       # missing
    {"mailbag_id": "B3", "file_received_at": 18 * HOUR},  # late
    {"mailbag_id": "B4"},                                 # nothing recorded
]
arrival_of = {f"B{n}": VAN_BACK for n in range(1, 5)}

late = processing.late_file_bags(requests, arrival_of)
for r in requests:
    when = r.get("file_received_at", "not recorded")
    print(f"{r['mailbag_id']}  file {when!s:<12} "
          f"{'§5.1.1 slow order' if r['mailbag_id'] in late else 'geocoded already'}")

assert late == {"B2", "B3"}, "a missing field is silence, not a missing file"

## What late costs

§5.2.2 is free for the other bags because it already happened. For these two it
becomes a fourth server between reconciliation and the clean room, and
`Readiness.late_ready` is not stored — it *is* having passed that stage, which
§5.2.2 puts on the critical path in exactly this case and no other.

In [ ]:
envelopes = [{"package_id": f"{bag}-{i}", "mailbag_id": bag,
              "package_type": "letter"}
             for bag in ("B1", "B2", "B3", "B4") for i in range(2)]
cleared = processing.schedule(envelopes, arrival_of, late_files=late)

print(f"{'':<8}{'reconciled':>11}{'geocoded':>10}{'sorted':>8}{'ready':>7}   flag")
for r in cleared:
    geocoded = "-" if r.geocoded_at is None else f"+{r.geocoded_at - VAN_BACK}"
    print(f"{r.package_id:<8}{f'+{r.reconciled_at - VAN_BACK}':>11}{geocoded:>10}"
          f"{f'+{r.sorted_at - VAN_BACK}':>8}{f'+{r.ready_at - VAN_BACK}':>7}"
          f"   {'late-ready' if r.late_ready else ''}")

assert {r.package_id[:2] for r in cleared if r.late_ready} == {"B2", "B3"}

# `schedule` returns them "in the order they clear the hub", and the paragraph
# below is about that order rather than about any one time. Pinned, because the
# sentence would go on reading well if the sort were by package_id.
assert [r.package_id for r in cleared] == [
    "B1-0", "B1-1", "B2-0", "B2-1", "B3-0", "B4-0", "B4-1", "B3-1"]

Read the last column against the first.

`B4-1` is the last envelope of the last bag and it is **not** the last one
ready: `B3-1` is, four seconds behind it, because `B3` went through a stage
`B4` did not. A late file does not only delay its own bag — it moves that bag
behind envelopes that reached the hub after it.

That is worth saying out loud because §5.3 loads vans against ready times.
An envelope that misses tonight's line-haul because its file was late is
§5.1.1's cost, and it is paid at a stage that has nothing to do with the van.

## A throughput is not a service time (§8.3)

If each step were a flat per-envelope delay, the hub would be infinitely
parallel: 4,800 envelopes arriving at once would all be ready a few minutes
later, and §5.2.3's clean room — which §8.3 names the likely bottleneck after
van-hours — could never be one.

Each step is therefore a single server, and an envelope starts when both it and
the server are free. The consequence is visible in one table: hold the arrival
fixed, hold the count fixed, and move only how many envelopes §5.2.3 claims.

In [ ]:
def last_ready(count, through_clean_room):
    """Minutes after the van is back before the whole pool is Ready (§5.2.6)."""
    pool = [{"package_id": f"P{i:03d}", "mailbag_id": "B1",
             "package_type": "assembly" if i < through_clean_room else "letter"}
            for i in range(count)]
    done = processing.schedule(pool, {"B1": VAN_BACK})
    return (max(r.ready_at for r in done) - VAN_BACK) / 60


minutes = [last_ready(200, n) for n in (0, 20, 40, 80, 200)]
for wanted, taken in zip((0, 20, 40, 80, 200), minutes, strict=True):
    print(f"{wanted:>4} of 200 through the clean room -> {taken:>5.1f} min")

# Twenty is where the two servers cost the same: 200 reconciliations at
# 1,200/hour is 600 s, and 20 assemblies at 120/hour is also 600 s. Below it
# §5.2.1 binds and the clean room is free; above it the line is the clean
# room's and nothing else in §5.2 matters. Pinned so the paragraph under here
# cannot outlive a change to `docs/assumptions.md`.
assert [round(m) for m in minutes] == [10, 10, 20, 40, 100]

Twenty of two hundred is the crossover, and it is not a coincidence: 200
reconciliations at 1,200/hour and 20 assemblies at 120/hour are both 600
seconds. Below it §5.2.1 sets the pace and the clean room is free. Above it
the clean room *is* the day — doubling its share doubles the wait, and the
other three steps stop mattering.

Which is §8.3's claim, reproduced rather than quoted. The claim is about the
real operation; this is about four invented rates producing the same shape,
and `docs/capacity-finding.md` is why those are not the same statement.

Open Question 4 is what would make them one.